# Agri-Link · Crop-Quality & Produce Model Benchmark (Colab GPU)

Trains and compares **Custom CNN, VGG16, ResNet50, MobileNetV2, EfficientNetB0** on the *Wild Rice Kernel Quality* dataset (CC-BY-4.0) and *FruitNet + Tomato + Freshness* datasets.
Picks the best model by **validation macro-F1**, evaluates on the untouched test split, and exports production `.keras` + `.json` models for `ml_backend/models/`.

> **Setup:** In the top menu, select **Runtime → Change runtime type → T4 GPU** before running.

## 1. Clone the Agri-Link Repository & Install Dependencies

In [ ]:
import os
%cd /content
if not os.path.isdir('/content/agri-link'):
    !git clone https://github.com/SaiNikhil6305/agrilinkk.git agri-link
else:
    !git -C /content/agri-link pull
!pip -q install pyarrow huggingface_hub scikit-learn matplotlib seaborn
%cd /content/agri-link


## 2. Download Wild Rice Kernel Dataset (16,632 kernels, ~5 GB)

In [ ]:
%cd /content/agri-link
from huggingface_hub import hf_hub_download
import os, shutil
os.makedirs('datasets/raw/wildrice', exist_ok=True)
for i in range(11):
    p = hf_hub_download('Project-AgML/wild_rice_kernel_quality_classification', f'raw/train-{i:05d}-of-00011.parquet', repo_type='dataset')
    shutil.copy(p, f'datasets/raw/wildrice/raw-{i:05d}.parquet')
!ls -lh datasets/raw/wildrice


## 3. Build Grain Quality Dataset (Premium / Good / Fair)
Maps: Premium = healthy; Good = minor defects; Fair = severe defects/unhulled. Applies 16x16 perceptual hash deduplication to eliminate data leakage.

In [ ]:
%cd /content/agri-link
!sed -i "s/CAP = {n: (300 if GRADE\[n\] != 'Fair' else 150)/CAP = {n: (1800 if GRADE[n] != 'Fair' else 900)/" datasets/scripts/prepare_wild_rice.py
!rm -rf datasets/quality datasets/demo_quality
!python datasets/scripts/prepare_wild_rice.py datasets/raw/wildrice datasets/quality datasets/demo_quality
!for d in datasets/quality/*; do echo $d $(ls $d | wc -l); done


## 4. Benchmark 5 Architectures on Grain Quality

In [ ]:
%cd /content/agri-link/ml_backend
!python benchmark_models.py --data ../datasets/quality --epochs 40 \
  --dataset-name 'Wild Rice Kernel Quality (Univ. of Manitoba) via AgML' --dataset-license CC-BY-4.0 \
  --label-mapping 'Premium=healthy_large+healthy_medium; Good=broken_low+discoloured_low; Fair=broken_high+discoloured_high+insect_infected+unhulled'


In [ ]:
%cd /content/agri-link/ml_backend
from IPython.display import Image, Markdown, display
display(Markdown(open('models/quality_benchmark/BENCHMARK.md').read()))
display(Image('models/quality_benchmark/benchmark_comparison.png'))
display(Image('models/quality_benchmark/confusion_matrices.png'))


## 5. Verify Selected Model on Never-Seen Kernels & Segregate

In [ ]:
%cd /content/agri-link/ml_backend
!python crop_segregation.py ../datasets/demo_quality | tail -5
!ls -la ../crop_quality_results/*


## 6. Download Trained Grain Model Package

In [ ]:
%cd /content/agri-link/ml_backend/models
from google.colab import files
!zip -qr /content/quality_model.zip crop_quality_model_final* quality_benchmark
files.download('/content/quality_model.zip')


## 7. Fruits & Vegetables Quality Model (8 Crops)
Downloads FruitNet (7 Indian fruits, Good/Mixed/Bad, CC-BY-4.0), AgML Tomato quality (CC-BY-4.0), and the Apple/Banana/Orange freshness dataset (MIT).

In [ ]:
%cd /content/agri-link
from huggingface_hub import hf_hub_download
import os, shutil
R = '/content/agri-link/datasets/raw'
for sub in ('fruitnet', 'tomato', 'freshness'):
    os.makedirs(f'{R}/{sub}', exist_ok=True)
for i in range(9):
    shutil.copy(hf_hub_download('Project-AgML/FruitNet_quality_classification', f'data/train-{i:05d}-of-00009.parquet', repo_type='dataset'), f'{R}/fruitnet/p{i}.parquet')
shutil.copy(hf_hub_download('Project-AgML/tomato_quality_classification', 'raw/train-00000-of-00001.parquet', repo_type='dataset'), f'{R}/tomato/raw.parquet')
for f in ('train', 'test'):
    shutil.copy(hf_hub_download('jojogo9/freshness', f'dataset_100/{f}.parquet', repo_type='dataset'), f'{R}/freshness/{f}.parquet')
print('All produce dataset shards downloaded successfully to', R)


In [ ]:
%cd /content/agri-link
!rm -rf datasets/produce datasets/demo_produce
!CAP=400 python datasets/scripts/prepare_produce.py datasets/raw datasets/produce datasets/demo_produce

%cd /content/agri-link/ml_backend
!python benchmark_models.py --data ../datasets/produce --name produce_quality_model --bench-dir produce_benchmark --epochs 40 \
  --supported-crops 'Apple,Banana,Guava,Lemon,Lime,Orange,Pomegranate,Tomato' \
  --dataset-name 'FruitNet + AgML Tomato quality + jojogo9/freshness' --dataset-license 'CC-BY-4.0 / CC-BY-4.0 / MIT' \
  --label-mapping 'Premium=Good/fresh; Good=FruitNet Mixed; Fair=Bad/rotten'

!python crop_segregation.py ../datasets/demo_produce | tail -5


In [ ]:
%cd /content/agri-link/ml_backend
from IPython.display import Image, Markdown, display
display(Markdown(open('models/produce_benchmark/BENCHMARK.md').read()))
display(Image('models/produce_benchmark/benchmark_comparison.png'))
display(Image('models/produce_benchmark/confusion_matrices.png'))


In [ ]:
%cd /content/agri-link/ml_backend/models
from google.colab import files
!zip -qr /content/produce_model.zip produce_quality_model* produce_benchmark
files.download('/content/produce_model.zip')
